[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafamayo/Workshop-UPNA-2026/blob/main/labs/day2/ai-mini-module/ai-notebook.ipynb)

# AI Mini-Module – Feature Extraction from FHIR Data
UPNA Workshop 2026 – Day 2

This notebook illustrates how structured FHIR data (e.g., Observations) can be used as input for an AI or ML model. We are **not** training a model here. Instead, we:

1. Query vital sign Observations for a Patient
2. Extract simple numerical features (e.g., last value, mean)
3. Discuss why standardization (FHIR + LOINC) matters for AI systems

⚠️ Use a Patient ID from your Day 1 labs, or enter one manually below.

In [ ]:
!pip install requests
# !pip install pandas
# when working on jupyter.org use this instead
!mamba install pandas

In [ ]:
import requests, json
import pandas as pd
import numpy as np

FHIR_SERVER = "https://hapi.fhir.org/baseR5/"
FHIR_SERVER

## 1. Choose a Patient ID
Use a Patient you created on Day 1. If that Patient has been deleted by the public server, try a different ID.

In [ ]:
patient_id = "856980"  # <-- Fill in manually
patient_id

## 2. Helper Functions
We define utility functions to:
- Fetch Observations by LOINC code
- Extract numeric values
- Convert them into a Pandas DataFrame for quick inspection

In [ ]:
def get_observations(patient_id, loinc_code):
    """
    Fetch Observations for the patient filtered by LOINC code.
    Returns a list of Observation resources.
    """
    url = f"{FHIR_SERVER}Observation?subject=Patient/{patient_id}&code={loinc_code}"
    r = requests.get(url, headers={"Accept": "application/fhir+json"})
    data = r.json()
    if "entry" not in data:
        return []
    return [e["resource"] for e in data["entry"]]


def extract_value(obs):
    """Try to extract a single numeric valueQuantity from an Observation."""
    try:
        return obs["valueQuantity"]["value"]
    except KeyError:
        return np.nan


def obs_to_dataframe(observations):
    """Convert a list of Observations to a tidy DataFrame."""
    rows = []
    for obs in observations:
        value = extract_value(obs)
        time = obs.get("effectiveDateTime", None)
        rows.append({"value": value, "time": time})
    return pd.DataFrame(rows)

## 3. Retrieve Vitals
We use common LOINC codes:
- Heart rate: `8867-4`
- Temperature: `8310-5`

You may add more later.

In [ ]:
# Replace patient_id above before running this

heart_rate_obs = get_observations(patient_id, "8867-4")
temperature_obs = get_observations(patient_id, "8310-5")

len(heart_rate_obs), len(temperature_obs)

### Convert to DataFrames

In [ ]:
df_hr = obs_to_dataframe(heart_rate_obs)
df_temp = obs_to_dataframe(temperature_obs)

df_hr.head(), df_temp.head()

## 4. Feature Extraction
We now compute simple **AI-ready features**:
- Last recorded value
- Mean value
- Standard deviation

These features are commonly used in time-series ML models (e.g., predicting deterioration).

In [ ]:
def compute_features(df):
    if df.empty:
        return {"last": np.nan, "mean": np.nan, "std": np.nan}
    return {
        "last": df.sort_values("time").value.iloc[-1],
        "mean": df.value.mean(),
        "std": df.value.std()
    }

features_hr = compute_features(df_hr)
features_temp = compute_features(df_temp)

features_hr, features_temp

## 5. Discussion
Questions to reflect on:

1. Why do AI models require **standardized** numerical inputs?
2. What could go wrong if Observations use incorrect coding systems?
3. What are possible sources of **bias** in this dataset?
4. How would you extend this feature extractor to support other vitals?

Write a short answer below if you wish.

In [ ]:
# Write your notes / answers here


## 6. Optional: Combine Features into a Single Feature Vector
This simulates a typical ML input row.

In [ ]:
feature_vector = {
    "hr_last": features_hr["last"],
    "hr_mean": features_hr["mean"],
    "hr_std": features_hr["std"],
    "temp_last": features_temp["last"],
    "temp_mean": features_temp["mean"],
    "temp_std": features_temp["std"]
}
feature_vector